## Importing all the Libraries

In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

## Numerical Columns
**Target Column** - Price_per_SqFt  
**Numerical Columns**  
*(int64)*
1. ID              : Should be removed
2. BHK             : Data ranges from 1 - 5 all having median price similar
3. Size_in_SqFt    : Debatable
4. Year_Built      : Can be removed as we already have age of property
5. Floor_No        : issue
6. Total_Floors    : issue
7. Age_of_Property : Range from 2 - 35 whole numbers
8. Nearby_Schools  : Data range from 1 - 10
9. Nearby_Hospitals: Data range from 1 - 10

*(float64)*
1. Price_in_Lakhs  : Target Column
2. Price_in_SqFt   : Can be removed as we already have price_in_Lakhs

**Note** - No outliers exists in the numerical columns

## Categorical Columns

**Categorical Columns**
1. State                              : 20 different states (Target encoding)
2. City                               : 42 different cities (Target encoding)
3. Locality                           : high cardinality (500), one hot, ordinal encoding not much effective (frequency encoding) 
4. Property_Type                      : (3 types) (one-hot encoding)
5. Furnished_Status                   : (3 types) hiarchical (ordinal encoding)
6. Public_Transport_Accessibility     : (3 types) hiarchical (ordinal encoding)
7.  Parking_Space                     : Yes/No (binary)
8.  Security                          : Yes/No (binary)
9.  Amenities                         : can be changed to numerical [5 amenities]
10.  Facing                           : 4 directions (ordinal encoding)
11.  Owner_Type                       : Broker builder owner can be dropped
12.  Availability_Status              : under construction ,ready (ordinal encoding)

## Working with New Dataset

In [2]:
# Importing Libraries
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import TargetEncoder

In [3]:
# Displaying the dataset
df = pd.read_csv("../dataset/melb_data.csv",thousands=",")
df.tail()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
13575,Wheelers Hill,12 Strada Cr,4,h,1245000.0,S,Barry,26/08/2017,16.7,3150.0,...,2.0,2.0,652.0,NaN,1981.0,NaN,-37.90562,145.16761,South-Eastern Metropolitan,7392.0
13576,Williamstown,77 Merrett Dr,3,h,1031000.0,SP,Williams,26/08/2017,6.8,3016.0,...,2.0,2.0,333.0,133.0,1995.0,NaN,-37.85927,144.87904,Western Metropolitan,6380.0
13577,Williamstown,83 Power St,3,h,1170000.0,S,Raine,26/08/2017,6.8,3016.0,...,2.0,4.0,436.0,NaN,1997.0,NaN,-37.85274,144.88738,Western Metropolitan,6380.0
13578,Williamstown,96 Verdon St,4,h,2500000.0,PI,Sweeney,26/08/2017,6.8,3016.0,...,1.0,5.0,866.0,157.0,1920.0,NaN,-37.85908,144.89299,Western Metropolitan,6380.0
13579,Yarraville,6 Agnes St,4,h,1285000.0,SP,Village,26/08/2017,6.3,3013.0,...,1.0,1.0,362.0,112.0,1920.0,NaN,-37.81188,144.88449,Western Metropolitan,6543.0


In [4]:
numerical_cols = df.select_dtypes(include=["number"]).columns
categorical_cols = df.select_dtypes(include=["object", "string"]).columns

categorical_cols

Index(['Suburb', 'Address', 'Type', 'Method', 'SellerG', 'Date', 'CouncilArea',
       'Regionname'],
      dtype='str')

In [5]:
df[categorical_cols].isnull().sum()

Suburb            0
Address           0
Type              0
Method            0
SellerG           0
Date              0
CouncilArea    1369
Regionname        0
dtype: int64

In [6]:
## Filling the NaN with Unknown as new value:
df["CouncilArea"] = df["CouncilArea"].fillna("Unknown")

In [7]:
# Dropping Not so important columns:
df = df.drop(columns=["Address", "SellerG"])

# Renaming the Categories in Type and method columns

In [8]:
df["Type"] = df["Type"].replace({
    "h": "House",
    "u": "Unit",
    "t": "Townhouse"
})

In [9]:
df["Method"] = df["Method"].replace({
    "S": "Sold",
    "SP": "Sold_Prior",
    "PI": "Passed_In",
    "VB": "Vendor_Bid",
    "SA": "Sold_After"
})

In [10]:
df.rename(columns={"Type": "Property_Type","Method": "Sale_Method"}, inplace=True)

In [11]:
#ONE HOT ENCODING
low_card_cols = ["Property_Type","Sale_Method","Regionname"]
one_hot = OneHotEncoder(handle_unknown="ignore",sparse_output=False).set_output(transform="pandas")
df_transformed_ohe = one_hot.fit_transform(df[low_card_cols])

In [12]:
df_transformed_ohe.head()

,Property_Type_House,Property_Type_Townhouse,Property_Type_Unit,Sale_Method_Passed_In,Sale_Method_Sold,Sale_Method_Sold_After,Sale_Method_Sold_Prior,Sale_Method_Vendor_Bid,Regionname_Eastern Metropolitan,Regionname_Eastern Victoria,Regionname_Northern Metropolitan,Regionname_Northern Victoria,Regionname_South-Eastern Metropolitan,Regionname_Southern Metropolitan,Regionname_Western Metropolitan,Regionname_Western Victoria
0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [13]:
# FREQUENCY ENCODING
suburb_freq = df["Suburb"].value_counts(normalize=True)

In [14]:
df["Suburb_freq"] = df["Suburb"].map(suburb_freq)
df.drop(columns=[c], inplace=True)

In [15]:
council_freq = df["CouncilArea"].value_counts(normalize=True)

In [16]:
df["CouncilArea_freq"] = df["CouncilArea"].map(council_freq)
df.drop(columns=["CouncilArea"], inplace=True)

In [17]:
df

,Rooms,Property_Type,Price,Sale_Method,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Regionname,Propertycount,Suburb_freq,CouncilArea_freq
0,2,House,1480000.0,Sold,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,NaN,NaN,-37.79960,144.99840,Northern Metropolitan,4019.0,0.004124,0.047644
1,2,House,1035000.0,Sold,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,-37.80790,144.99340,Northern Metropolitan,4019.0,0.004124,0.047644
2,3,House,1465000.0,Sold_Prior,4/03/2017,2.5,3067.0,3.0,2.0,0.0,134.0,150.0,1900.0,-37.80930,144.99440,Northern Metropolitan,4019.0,0.004124,0.047644
3,3,House,850000.0,Passed_In,4/03/2017,2.5,3067.0,3.0,2.0,1.0,94.0,NaN,NaN,-37.79690,144.99690,Northern Metropolitan,4019.0,0.004124,0.047644
4,4,House,1600000.0,Vendor_Bid,4/06/2016,2.5,3067.0,3.0,1.0,2.0,120.0,142.0,2014.0,-37.80720,144.99410,Northern Metropolitan,4019.0,0.004124,0.047644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,4,House,1245000.0,Sold,26/08/2017,16.7,3150.0,4.0,2.0,2.0,652.0,NaN,1981.0,-37.90562,145.16761,South-Eastern Metropolitan,7392.0,0.001915,0.100810
13576,3,House,1031000.0,Sold_Prior,26/08/2017,6.8,3016.0,3.0,2.0,2.0,333.0,133.0,1995.0,-37.85927,144.87904,Western Metropolitan,6380.0,0.007879,0.100810
13577,3,House,1170000.0,Sold,26/08/2017,6.8,3016.0,3.0,2.0,4.0,436.0,NaN,1997.0,-37.85274,144.88738,Western Metropolitan,6380.0,0.007879,0.100810
13578,4,House,2500000.0,Passed_In,26/08/2017,6.8,3016.0,4.0,1.0,5.0,866.0,157.0,1920.0,-37.85908,144.89299,Western Metropolitan,6380.0,0.007879,0.100810


***Date column will be taken care of in Numerical column transformation***

# Transforamtion of Numerical columns

## Checking for Null Values

In [18]:
df[numerical_cols].isnull().sum()

Rooms               0
Price               0
Distance            0
Postcode            0
Bedroom2            0
Bathroom            0
Car                62
Landsize            0
BuildingArea     6450
YearBuilt        5375
Lattitude           0
Longtitude          0
Propertycount       0
dtype: int64

Here we have Car - we will fill with median  
BuildingArea - almost 50% missing  
             - important column so we cannot drow it.  
             - Dropping rows will lead to loss of half of the data  
             - using median + an indicator that the data was missing  
  
YearBuilt    - Same as BuildingArea  

In [19]:
# Car
df["Car"] = df["Car"].fillna(df["Car"].median())

# BuildingArea
df["BuildingArea_missing"] = df["BuildingArea"].isnull().astype(int)
df["BuildingArea"] = df["BuildingArea"].fillna(df["BuildingArea"].median())

# YearBuilt
df["YearBuilt_missing"] = df["YearBuilt"].isnull().astype(int)
df["YearBuilt"] = df["YearBuilt"].fillna(df["YearBuilt"].median())

In [20]:
df

,Rooms,Property_Type,Price,Sale_Method,Date,Distance,Postcode,Bedroom2,Bathroom,Car,...,BuildingArea,YearBuilt,Lattitude,Longtitude,Regionname,Propertycount,Suburb_freq,CouncilArea_freq,BuildingArea_missing,YearBuilt_missing
0,2,House,1480000.0,Sold,3/12/2016,2.5,3067.0,2.0,1.0,1.0,...,126.0,1970.0,-37.79960,144.99840,Northern Metropolitan,4019.0,0.004124,0.047644,1,1
1,2,House,1035000.0,Sold,4/02/2016,2.5,3067.0,2.0,1.0,0.0,...,79.0,1900.0,-37.80790,144.99340,Northern Metropolitan,4019.0,0.004124,0.047644,0,0
2,3,House,1465000.0,Sold_Prior,4/03/2017,2.5,3067.0,3.0,2.0,0.0,...,150.0,1900.0,-37.80930,144.99440,Northern Metropolitan,4019.0,0.004124,0.047644,0,0
3,3,House,850000.0,Passed_In,4/03/2017,2.5,3067.0,3.0,2.0,1.0,...,126.0,1970.0,-37.79690,144.99690,Northern Metropolitan,4019.0,0.004124,0.047644,1,1
4,4,House,1600000.0,Vendor_Bid,4/06/2016,2.5,3067.0,3.0,1.0,2.0,...,142.0,2014.0,-37.80720,144.99410,Northern Metropolitan,4019.0,0.004124,0.047644,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,4,House,1245000.0,Sold,26/08/2017,16.7,3150.0,4.0,2.0,2.0,...,126.0,1981.0,-37.90562,145.16761,South-Eastern Metropolitan,7392.0,0.001915,0.100810,1,0
13576,3,House,1031000.0,Sold_Prior,26/08/2017,6.8,3016.0,3.0,2.0,2.0,...,133.0,1995.0,-37.85927,144.87904,Western Metropolitan,6380.0,0.007879,0.100810,0,0
13577,3,House,1170000.0,Sold,26/08/2017,6.8,3016.0,3.0,2.0,4.0,...,126.0,1997.0,-37.85274,144.88738,Western Metropolitan,6380.0,0.007879,0.100810,1,0
13578,4,House,2500000.0,Passed_In,26/08/2017,6.8,3016.0,4.0,1.0,5.0,...,157.0,1920.0,-37.85908,144.89299,Western Metropolitan,6380.0,0.007879,0.100810,0,0


In [21]:
# Extracting Useful information from Date (date of sale col)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

In [22]:
df["Sale_Year"] = df["Date"].dt.year
df["Sale_Month"] = df["Date"].dt.month

df["Property_Age_At_Sale"] = df["Sale_Year"] - df["YearBuilt"]

In [23]:
df.drop(columns=["Date"], inplace=True)

In [24]:
df

,Rooms,Property_Type,Price,Sale_Method,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,...,Longtitude,Regionname,Propertycount,Suburb_freq,CouncilArea_freq,BuildingArea_missing,YearBuilt_missing,Sale_Year,Sale_Month,Property_Age_At_Sale
0,2,House,1480000.0,Sold,2.5,3067.0,2.0,1.0,1.0,202.0,...,144.99840,Northern Metropolitan,4019.0,0.004124,0.047644,1,1,2016,12,46.0
1,2,House,1035000.0,Sold,2.5,3067.0,2.0,1.0,0.0,156.0,...,144.99340,Northern Metropolitan,4019.0,0.004124,0.047644,0,0,2016,2,116.0
2,3,House,1465000.0,Sold_Prior,2.5,3067.0,3.0,2.0,0.0,134.0,...,144.99440,Northern Metropolitan,4019.0,0.004124,0.047644,0,0,2017,3,117.0
3,3,House,850000.0,Passed_In,2.5,3067.0,3.0,2.0,1.0,94.0,...,144.99690,Northern Metropolitan,4019.0,0.004124,0.047644,1,1,2017,3,47.0
4,4,House,1600000.0,Vendor_Bid,2.5,3067.0,3.0,1.0,2.0,120.0,...,144.99410,Northern Metropolitan,4019.0,0.004124,0.047644,0,0,2016,6,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,4,House,1245000.0,Sold,16.7,3150.0,4.0,2.0,2.0,652.0,...,145.16761,South-Eastern Metropolitan,7392.0,0.001915,0.100810,1,0,2017,8,36.0
13576,3,House,1031000.0,Sold_Prior,6.8,3016.0,3.0,2.0,2.0,333.0,...,144.87904,Western Metropolitan,6380.0,0.007879,0.100810,0,0,2017,8,22.0
13577,3,House,1170000.0,Sold,6.8,3016.0,3.0,2.0,4.0,436.0,...,144.88738,Western Metropolitan,6380.0,0.007879,0.100810,1,0,2017,8,20.0
13578,4,House,2500000.0,Passed_In,6.8,3016.0,4.0,1.0,5.0,866.0,...,144.89299,Western Metropolitan,6380.0,0.007879,0.100810,0,0,2017,8,97.0


In [25]:
print("YearBuilt < 1800 :", (df["YearBuilt"] < 1800).sum())
print("YearBuilt > 2025 :", (df["YearBuilt"] > 2025).sum())

YearBuilt < 1800 : 1
YearBuilt > 2025 : 0


In [26]:
df.loc[df["YearBuilt"] < 1800, "YearBuilt"] = None
df["YearBuilt"] = df["YearBuilt"].fillna(df["YearBuilt"].median())

In [27]:
outlier_counts = {}

for col in numerical_cols:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3-Q1
    
    lower_bound = Q1-1.5 * IQR
    upper_bound = Q3+1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    outlier_counts[col] = len(outliers)

outlier_counts

{'Rooms': 682,
 'Price': 612,
 'Distance': 411,
 'Postcode': 208,
 'Bedroom2': 655,
 'Bathroom': 143,
 'Car': 644,
 'Landsize': 368,
 'BuildingArea': 5565,
 'YearBuilt': 3974,
 'Lattitude': 262,
 'Longtitude': 408,
 'Propertycount': 359}

In [28]:
for col in ["BuildingArea", "Landsize"]:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    df[col] = df[col].clip(lower, upper)

In [29]:
df

,Rooms,Property_Type,Price,Sale_Method,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,...,Longtitude,Regionname,Propertycount,Suburb_freq,CouncilArea_freq,BuildingArea_missing,YearBuilt_missing,Sale_Year,Sale_Month,Property_Age_At_Sale
0,2,House,1480000.0,Sold,2.5,3067.0,2.0,1.0,1.0,202.0,...,144.99840,Northern Metropolitan,4019.0,0.004124,0.047644,1,1,2016,12,46.0
1,2,House,1035000.0,Sold,2.5,3067.0,2.0,1.0,0.0,156.0,...,144.99340,Northern Metropolitan,4019.0,0.004124,0.047644,0,0,2016,2,116.0
2,3,House,1465000.0,Sold_Prior,2.5,3067.0,3.0,2.0,0.0,134.0,...,144.99440,Northern Metropolitan,4019.0,0.004124,0.047644,0,0,2017,3,117.0
3,3,House,850000.0,Passed_In,2.5,3067.0,3.0,2.0,1.0,94.0,...,144.99690,Northern Metropolitan,4019.0,0.004124,0.047644,1,1,2017,3,47.0
4,4,House,1600000.0,Vendor_Bid,2.5,3067.0,3.0,1.0,2.0,120.0,...,144.99410,Northern Metropolitan,4019.0,0.004124,0.047644,0,0,2016,6,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,4,House,1245000.0,Sold,16.7,3150.0,4.0,2.0,2.0,652.0,...,145.16761,South-Eastern Metropolitan,7392.0,0.001915,0.100810,1,0,2017,8,36.0
13576,3,House,1031000.0,Sold_Prior,6.8,3016.0,3.0,2.0,2.0,333.0,...,144.87904,Western Metropolitan,6380.0,0.007879,0.100810,0,0,2017,8,22.0
13577,3,House,1170000.0,Sold,6.8,3016.0,3.0,2.0,4.0,436.0,...,144.88738,Western Metropolitan,6380.0,0.007879,0.100810,1,0,2017,8,20.0
13578,4,House,2500000.0,Passed_In,6.8,3016.0,4.0,1.0,5.0,866.0,...,144.89299,Western Metropolitan,6380.0,0.007879,0.100810,0,0,2017,8,97.0


In [30]:
# Dropping Redundant and less usefull columns:
df.drop(columns=["Postcode", "Bedroom2"], inplace=True)

In [31]:
df

,Rooms,Property_Type,Price,Sale_Method,Distance,Bathroom,Car,Landsize,BuildingArea,YearBuilt,...,Longtitude,Regionname,Propertycount,Suburb_freq,CouncilArea_freq,BuildingArea_missing,YearBuilt_missing,Sale_Year,Sale_Month,Property_Age_At_Sale
0,2,House,1480000.0,Sold,2.5,1.0,1.0,202.0,126.00,1970.0,...,144.99840,Northern Metropolitan,4019.0,0.004124,0.047644,1,1,2016,12,46.0
1,2,House,1035000.0,Sold,2.5,1.0,0.0,156.0,110.09,1900.0,...,144.99340,Northern Metropolitan,4019.0,0.004124,0.047644,0,0,2016,2,116.0
2,3,House,1465000.0,Sold_Prior,2.5,2.0,0.0,134.0,141.85,1900.0,...,144.99440,Northern Metropolitan,4019.0,0.004124,0.047644,0,0,2017,3,117.0
3,3,House,850000.0,Passed_In,2.5,2.0,1.0,94.0,126.00,1970.0,...,144.99690,Northern Metropolitan,4019.0,0.004124,0.047644,1,1,2017,3,47.0
4,4,House,1600000.0,Vendor_Bid,2.5,1.0,2.0,120.0,141.85,2014.0,...,144.99410,Northern Metropolitan,4019.0,0.004124,0.047644,0,0,2016,6,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,4,House,1245000.0,Sold,16.7,2.0,2.0,652.0,126.00,1981.0,...,145.16761,South-Eastern Metropolitan,7392.0,0.001915,0.100810,1,0,2017,8,36.0
13576,3,House,1031000.0,Sold_Prior,6.8,2.0,2.0,333.0,133.00,1995.0,...,144.87904,Western Metropolitan,6380.0,0.007879,0.100810,0,0,2017,8,22.0
13577,3,House,1170000.0,Sold,6.8,2.0,4.0,436.0,126.00,1997.0,...,144.88738,Western Metropolitan,6380.0,0.007879,0.100810,1,0,2017,8,20.0
13578,4,House,2500000.0,Passed_In,6.8,1.0,5.0,866.0,141.85,1920.0,...,144.89299,Western Metropolitan,6380.0,0.007879,0.100810,0,0,2017,8,97.0


In [32]:
numerical_cols = df.select_dtypes(include=["number"]).columns
df[numerical_cols].skew().sort_values(ascending=False)

Price                   2.239624
Property_Age_At_Sale    2.178097
Distance                1.676937
Bathroom                1.377406
Car                     1.366017
Propertycount           1.069339
Suburb_freq             0.891942
Landsize                0.589711
YearBuilt_missing       0.426193
Rooms                   0.376478
BuildingArea_missing    0.100284
Sale_Month              0.098326
BuildingArea            0.002220
CouncilArea_freq       -0.072993
Sale_Year              -0.134041
Longtitude             -0.210991
Lattitude              -0.426695
YearBuilt              -0.876307
dtype: float64

In [36]:
df_transformed = pd.concat([df, df_transformed_ohe], axis=1)
df_transformed.drop(columns=["Property_Type","Sale_Method","Regionname"],inplace=True)

In [43]:
#df_transformed.to_csv("../dataset/df_transformed.csv", index=False)

In [37]:
skewness = df[numerical_cols].skew()

high_skew = skewness[abs(skewness) > 1]

print(high_skew.sort_values(ascending=False))

Price                   2.239624
Property_Age_At_Sale    2.178097
Distance                1.676937
Bathroom                1.377406
Car                     1.366017
Propertycount           1.069339
dtype: float64


In [39]:
from sklearn.preprocessing import StandardScaler

X = df_transformed.drop(columns=["Price"])
y = df_transformed["Price"]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [40]:
X

,Rooms,Distance,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount,...,Sale_Method_Sold_Prior,Sale_Method_Vendor_Bid,Regionname_Eastern Metropolitan,Regionname_Eastern Victoria,Regionname_Northern Metropolitan,Regionname_Northern Victoria,Regionname_South-Eastern Metropolitan,Regionname_Southern Metropolitan,Regionname_Western Metropolitan,Regionname_Western Victoria
0,2,2.5,1.0,1.0,202.0,126.00,1970.0,-37.79960,144.99840,4019.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,2,2.5,1.0,0.0,156.0,110.09,1900.0,-37.80790,144.99340,4019.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,3,2.5,2.0,0.0,134.0,141.85,1900.0,-37.80930,144.99440,4019.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,3,2.5,2.0,1.0,94.0,126.00,1970.0,-37.79690,144.99690,4019.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,4,2.5,1.0,2.0,120.0,141.85,2014.0,-37.80720,144.99410,4019.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,4,16.7,2.0,2.0,652.0,126.00,1981.0,-37.90562,145.16761,7392.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
13576,3,6.8,2.0,2.0,333.0,133.00,1995.0,-37.85927,144.87904,6380.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
13577,3,6.8,2.0,4.0,436.0,126.00,1997.0,-37.85274,144.88738,6380.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
13578,4,6.8,1.0,5.0,866.0,141.85,1920.0,-37.85908,144.89299,6380.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
